Before running the code below, I hypothesize that the model will have a significantly higher correlation of failures with machines that exceed their designed failure specification by plus or minus 10% above the max or below the minimum respectively. Given that one of the rules, OSF, utilizes features not present in the model, I believe it will struggle to accurately predict these failures moreso than with the other failure modes.

In [7]:
import numpy as np
import pandas as pd
import pipeline
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

# Cutoff and band width. Change BAND to widen the groups.
CUTOFF = 0.66
BAND = 0.10 # +/- 10%

# Same data, split, and model as confusion_matrix_numpy.
proc = pipeline.load_data("processed")
raw = pipeline.load_data("raw")

X = proc[pipeline.PREDICTORS]
y = proc[pipeline.TARGET]


In [8]:
# Create split of 2000 stratified data points for test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# train model
model = pipeline.create_model(X_train, y_train)

scores = model.predict_proba(X_test)[:, 1]

In [9]:
# copy only test rows from the raw file, with Type and flags.
t = raw.loc[y_test.index].copy()

# verify copy is still ordered properly
assert (t["Machine failure"].to_numpy() == y_test.to_numpy()).all()

# add scores based and flag alerts based on CUTOFF
t["score"] = scores
t["alert"] = t["score"] >= CUTOFF

# Should print 2000 245 68.
print("Machines:",len(t), "\nAlerts:", int(t["alert"].sum()),"\nFailures:", int(t["Machine failure"].sum()))

Machines: 2000 
Alerts: 245 
Failures: 68


In [10]:
# Establish vars and methods

# Rule columns. Convert rpm to rad/s for power.
t["power"] = t["Torque"] * (t["Rotational speed"] * 2 * np.pi / 60)
t["temp_diff"] = abs(t["Process temperature"] - t["Air temperature"])
t["strain"] = t["Tool wear"] * t["Torque"]
t["strain_limit"] = t["Type"].map({"L": 11000, "M": 12000, "H": 13000})

# Overstrain groups within margtin (BAND) of strain_limit.
over = (t["strain"] > t["strain_limit"]) & (t["strain"] <= (1 + BAND) * t["strain_limit"])
under = (t["strain"] <= t["strain_limit"]) & (t["strain"] >= (1 - BAND) * t["strain_limit"])

# Power groups within BAND of 3500W and 9000W.
power = t["power"]
low_fail = (power >= 3500 * (1 - BAND)) & (power < 3500)
low_safe = (power >= 3500) & (power <= 3500 * (1 + BAND))
high_safe = (power >= 9000 * (1 - BAND)) & (power <= 9000)
high_fail = (power > 9000) & (power <= 9000 * (1 + BAND))

# heat groups near 8.6 kelvin with rpm under 1380.
delta_k = t["temp_diff"]
slow = t["Rotational speed"] < 1380
heat_fail = slow & (delta_k >= 8.6 * (1 - BAND)) & (delta_k < 8.6)
heat_safe = slow & (delta_k >= 8.6) & (delta_k <= 8.6 * (1 + BAND))

groups = {
    "OSF_over": over,
    "OSF_under": under,
    "PWF_low_fail": low_fail,
    "PWF_low_safe": low_safe,
    "PWF_high_fail": high_fail,
    "PWF_high_safe": high_safe,
    "HDF_fail": heat_fail,
    "HDF_safe": heat_safe,
}

def summarize(name, m):
    g = t[m]
    fail = g["Machine failure"] == 1
    return {"group": name, "machines": len(g),
            "failures": int(fail.sum()),
            "alerts": int(g["alert"].sum()),
            "tp": int((g["alert"] & fail).sum()),
            "fp": int((g["alert"] & ~fail).sum()),
            "alert_rate": g["alert"].mean(),
            "mean_score": g["score"].mean()}

In [11]:
summary = pd.DataFrame([summarize(k, m) for k, m in groups.items()])
print(summary)

# save to CSV
filename = "near_rules_summary_" + str(BAND) + '_band.csv'
summary.to_csv(pipeline.OUTPUTS_DATA_DIR / filename, index=False)

# compare similarity of failure flags against ruleset. A result less than 1.0 indicates an issue in the application of the ruleset.
print("OSF", ((t["strain"] > t["strain_limit"]) == (t["OSF"] == 1)).mean())
print("PWF", (((power < 3500) | (power > 9000)) == (t["PWF"] == 1)).mean())
print("HDF", (((delta_k < 8.6) & slow) == (t["HDF"] == 1)).mean())

           group  machines  failures  alerts  tp  fp  alert_rate  mean_score
0       OSF_over        12        12      12  12   0    1.000000    0.864945
1      OSF_under        33         5      25   4  21    0.757576    0.758576
2   PWF_low_fail         2         2       2   2   0    1.000000    0.769960
3   PWF_low_safe        11         0       3   0   3    0.272727    0.395405
4  PWF_high_fail         6         6       5   5   0    0.833333    0.900600
5  PWF_high_safe        75        19      51  19  32    0.680000    0.745285
6       HDF_fail        28        28      21  21   0    0.750000    0.775020
7       HDF_safe        58         4      29   4  25    0.500000    0.583026
OSF 1.0
PWF 1.0
HDF 1.0


Each row above reports:
- the count of machines within each failure mode group's margins
- the number of actual failures
- the number of alerts triggered by the model
- the TP and FP counts
- the frequency of alerts
- the model's mean prediction score for each group.

Examining these results, all models under the threshold (under/safe) appear to exhibit notably lower alert rates compared to those groups exceeding the established failure mode rules. However we would need further grainularity within the datapoints to understand whether the model accurately understands the exact thresholds that cause failure based on the rules of the synthetic dataset.

Now that we've seen it work on available test data, we would like to observe how the model reacts to more stringent conditions.
Below, only data already close to the boundaries of the failure modes is identified. Those data points are then used as the basis to synthesize new data that has been manually shifted under or over the limit by 1% to see how granularly the model is able to identify the rules governing failure.

In [12]:
# reduce band tolerance
SMALL_BAND = 0.05
DELTA_PCT = 0.01

results = []

# OSF boundary (strain vs limit)
# find test row near strain limit
data_near_osf = t[abs(t["strain"] - t["strain_limit"]) / t["strain_limit"] < SMALL_BAND]
if not data_near_osf.empty:
    base = data_near_osf.iloc[0].copy()
    limit_val = base["strain_limit"]
    
    for sign in [-1, 1]:
        synth = base.copy()

        # push strain just below/above strain_limit
        target_strain = limit_val * (1 + sign * DELTA_PCT)
        synth["Tool wear"] = target_strain / synth["Torque"]
        synth["strain"] = synth["Tool wear"] * synth["Torque"]
        synth["alert"] = synth["strain"] > limit_val
        results.append(("OSF", sign, synth))

# PWF boundary (power at 3500W and 9000W)
for boundary_power in [3500, 9000]:
    data_near_pwf = t[abs(t["power"] - boundary_power) / boundary_power < SMALL_BAND]
    if not data_near_pwf.empty:
        base = data_near_pwf.iloc[0].copy()
        torque = base["Torque"]
        
        for sign in [-1, 1]:
            synth = base.copy()
            target_power = boundary_power * (1 + sign * DELTA_PCT)

            # adjust rotational speed to target power
            synth["Rotational speed"] = (target_power * 60) / (torque * 2 * np.pi)
            synth["power"] = torque * (synth["Rotational speed"] * 2 * np.pi / 60)

            # PWF triggers if power < 3500 or power > 9000
            synth["alert"] = (synth["power"] < 3500) | (synth["power"] > 9000)
            results.append((f"PWF_{'low' if boundary_power == 3500 else 'high'}", sign, synth))

# HDF boundary (delta_k at 8.6 with low RPM)
data_near_hdf = t[(t["Rotational speed"] < 1380) & (abs(t["temp_diff"] - 8.6) < 0.5)]
if not data_near_hdf.empty:
    base = data_near_hdf.iloc[0].copy()

    for sign in [-1, 1]:
        synth = base.copy()
        target_diff = 8.6 * (1 + sign * DELTA_PCT)

        # Adjust Process temperature keeping Air temperature constant
        synth["Process temperature"] = synth["Air temperature"] + target_diff
        synth["temp_diff"] = abs(synth["Process temperature"] - synth["Air temperature"])
        synth["alert"] = (synth["temp_diff"] < 8.6) & (synth["Rotational speed"] < 1380)
        results.append(("HDF", sign, synth))

# eval all synth rows through the model
records = []
for group_name, sign, synth_row in results:
    X_row = pd.DataFrame([synth_row])[pipeline.PREDICTORS]
    score = model.predict_proba(X_row)[0, 1]
    records.append({
        "group": group_name,
        "adjustment": "under boundary (-1%)" if sign < 0 else "over boundary (+1%)",
        "alert_expected": synth_row["alert"],
        "score": round(score, 4),
        "alert_reported": score >= CUTOFF
    })

# save and print
boundary_report = pd.DataFrame(records)

filename = "near_rules_boundary_report_" + str(DELTA_PCT) + '_delta.csv'
boundary_report.to_csv(pipeline.OUTPUTS_DATA_DIR / filename, index=False)

print(boundary_report.to_string(index=False))

   group           adjustment  alert_expected  score  alert_reported
     OSF under boundary (-1%)           False 0.7446            True
     OSF  over boundary (+1%)            True 0.7544            True
 PWF_low under boundary (-1%)            True 0.8331            True
 PWF_low  over boundary (+1%)           False 0.8917            True
PWF_high under boundary (-1%)           False 0.9727            True
PWF_high  over boundary (+1%)            True 0.9789            True
     HDF under boundary (-1%)            True 0.6741            True
     HDF  over boundary (+1%)           False 0.6356           False


The failure group with the lowest model score was HDF Above Boundary (+1%) at 0.6356. Interestingly this is the only test that both expected and reported no alert. Examining the other groups, it is clear that the model has not cleanly identified the exact correlation of conditions that govern the machine failures, regardless of mode of failure. 

With this in mind, I believe that in order to improve the model further, we need to adjust the features present to the model. Seeing as one of the modes of failure is dictated in part by the quality (type), we should reintroduce the one-hot encoded type features into the list of predictors. We could also utilize some of the features created in this examination such as the temperature delta, the power calculations and the strain of the models.